# 1. Imports

In [ ]:
import matplotlib.pyplot as plt 
import pandas
from pandas import DataFrame        
from IPython.display import Markdown, display 

# 2. Data Loading

In [3]:
dataset: DataFrame = pandas.read_csv("data/caso_full.csv")

## 2.1 Schema Overview

In [4]:
display(Markdown(f"**Amount of tuples**: {len(dataset)}"))

display(Markdown(f"**Column names and its types**:"))

columns_names_with_type: DataFrame = DataFrame(
    {
        "column_name": dataset.columns,
        "type": dataset.dtypes.values
    }    
)

columns_names_with_type

**Amount of tuples**: 3853648

**Column names and its types**:

,column_name,type
0,city,object
1,city_ibge_code,float64
2,date,object
3,epidemiological_week,int64
4,estimated_population,float64
5,estimated_population_2019,float64
6,is_last,bool
7,is_repeated,bool
8,last_available_confirmed,int64
9,last_available_confirmed_per_100k_inhabitants,float64


### Sample

In [5]:
dataset.head(10)

,city,city_ibge_code,date,epidemiological_week,estimated_population,estimated_population_2019,is_last,is_repeated,last_available_confirmed,last_available_confirmed_per_100k_inhabitants,last_available_date,last_available_death_rate,last_available_deaths,order_for_place,place_type,state,new_confirmed,new_deaths
0,Rio Branco,1200401.0,2020-03-17,202012,413418.0,407319.0,False,False,3,0.72566,2020-03-17,0.0,0,1,city,AC,3,0
1,NaN,12.0,2020-03-17,202012,894470.0,881935.0,False,False,3,0.33539,2020-03-17,0.0,0,1,state,AC,3,0
2,Rio Branco,1200401.0,2020-03-18,202012,413418.0,407319.0,False,False,3,0.72566,2020-03-18,0.0,0,2,city,AC,0,0
3,NaN,12.0,2020-03-18,202012,894470.0,881935.0,False,False,3,0.33539,2020-03-18,0.0,0,2,state,AC,0,0
4,Rio Branco,1200401.0,2020-03-19,202012,413418.0,407319.0,False,False,4,0.96754,2020-03-19,0.0,0,3,city,AC,1,0
5,NaN,12.0,2020-03-19,202012,894470.0,881935.0,False,False,4,0.44719,2020-03-19,0.0,0,3,state,AC,1,0
6,Rio Branco,1200401.0,2020-03-20,202012,413418.0,407319.0,False,False,7,1.69320,2020-03-20,0.0,0,4,city,AC,3,0
7,NaN,12.0,2020-03-20,202012,894470.0,881935.0,False,False,7,0.78259,2020-03-20,0.0,0,4,state,AC,3,0
8,Rio Branco,1200401.0,2020-03-21,202012,413418.0,407319.0,False,False,11,2.66075,2020-03-21,0.0,0,5,city,AC,4,0
9,NaN,12.0,2020-03-21,202012,894470.0,881935.0,False,False,11,1.22978,2020-03-21,0.0,0,5,state,AC,4,0


### Memory Usage Before Optimization

In [6]:
dataset_total_memory_usage = dataset.memory_usage(deep = True).sum() / (1024 ** 2)

display(
    Markdown(
        f"**Total memory usage:** {dataset_total_memory_usage:.2f} MB"
    )
)

**Total memory usage:** 1553.11 MB

# 3. Memory Optimization

In [ ]:
def optimize_dtypes(df: DataFrame) -> tuple[DataFrame, DataFrame]:
    conversions = {
        "city":                      ("category", None),
        "city_ibge_code":            ("Int32",    lambda c: c.round()),
        "date":                      ("datetime", None),
        "last_available_date":       ("datetime", None),
        "estimated_population":      ("Int32",    None),
        "estimated_population_2019": ("Int32",    None),
        "place_type":                ("category", lambda c: c.replace({"city": "C", "state": "S"})),
    }
    rows = []
    for col, (dtype, pre) in conversions.items():
        before = df[col].memory_usage(deep=True) / 1024**2
        if pre:
            df[col] = pre(df[col])
        if dtype == "datetime":
            df[col] = pandas.to_datetime(df[col])
        else:
            df[col] = df[col].astype(dtype)
        after = df[col].memory_usage(deep=True) / 1024**2
        rows.append({
            "column":      col,
            "before_MB":   round(before, 3),
            "after_MB":    round(after, 3),
            "reduction_%": round((before - after) / before * 100, 1),
        })
    return df, DataFrame(rows)

dataset, memory_report = optimize_dtypes(dataset)
memory_report

## Memory Usage After Optimization

In [21]:
dataset_total_memory_usage_after_optimization = dataset.memory_usage(deep = True).sum() / (1024 ** 2)

display(
    Markdown(
        f"**Total memory usage:** {dataset_total_memory_usage_after_optimization:.2f} MB"
    )
)

**Total memory usage:** 584.87 MB

In [ ]:
before = dataset_total_memory_usage
after  = dataset_total_memory_usage_after_optimization
saved  = before - after
reduction = (saved / before) * 100

labels = ["Before Optimization", "After Optimization"]
values = [before, after]

plt.figure(figsize=(8, 5))

bars = plt.bar(labels, values)

plt.title("Dataset Memory Usage Optimization")
plt.ylabel("Memory Usage (MB)")

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{height:.0f} MB",
        ha="center",
        va="bottom",
        fontsize=11,
    )

plt.text(
    0.5,
    max(values) * 0.6,
    f"Memory reduced by {reduction:.2f}%\n({saved:.0f} MB saved)",
    ha="center",
    fontsize=12,
)

plt.show()

# 4. Missing Values Analysis

## Summary

In [23]:
display(
    Markdown(f"**Total amount of missing values**: {dataset.isna().sum().sum()}")
)

**Total amount of missing values**: 90223

## Per-Column Detail

In [24]:
empty_data_reckoning_rendered: DataFrame = DataFrame(dataset.isna().sum().sort_values(ascending = True))
empty_data_reckoning_rendered

,0
last_available_confirmed,0
state,0
place_type,0
order_for_place,0
last_available_deaths,0
last_available_death_rate,0
last_available_date,0
new_confirmed,0
new_deaths,0
is_last,0


### city

In [25]:
missing_city_mask = (dataset["city"].isna()) | (dataset["city"] == "Importados/Indefinidos")

empty_city_values = dataset.loc[
    missing_city_mask,
    ["city", "state", "city_ibge_code"]
]

empty_city_values

,city,state,city_ibge_code
1,NaN,AC,12
3,NaN,AC,12
5,NaN,AC,12
7,NaN,AC,12
9,NaN,AC,12
...,...,...,...
3853087,NaN,TO,17
3853227,NaN,TO,17
3853367,NaN,TO,17
3853507,NaN,TO,17


In [26]:
missing_city_rate: float = len(empty_city_values) / len(dataset)

display(
    Markdown(
        f"**Missing city rate:** {missing_city_rate:.2f}%"
    )
)

display(
    Markdown(
        "**So... drop it!**" if missing_city_rate < 0.25 else "**Consider imputing values**"
    )
)

**Missing city rate:** 0.01%

**So... drop it!**

### last_available_confirmed_per_100k_inhabitants

In [ ]:
missing_last_available_per_100k_inhabitants_mask = dataset["last_available_confirmed_per_100k_inhabitants"].isna()

empty_values_last_100k = dataset.loc[
    missing_last_available_per_100k_inhabitants_mask,
    ["last_available_confirmed_per_100k_inhabitants", "city", "state", "city_ibge_code"]
]

empty_values_last_100k

,city,state,city_ibge_code,estimated_population
431,Manoel Urbano,AC,1200344,9581
436,Porto Walter,AC,1200393,12241
453,Manoel Urbano,AC,1200344,9581
458,Porto Walter,AC,1200393,12241
475,Manoel Urbano,AC,1200344,9581


### estimated_population_2019

In [28]:
missing_estimated_population_2019_mask = (dataset["estimated_population_2019"].isna() | dataset["estimated_population_2019"] == "<NA>")

estimated_population = dataset.loc[
    missing_estimated_population_2019_mask,
    ["estimated_population", "estimated_population_2019", "city", "state", "city_ibge_code"]
]

estimated_population

,estimated_population,estimated_population_2019,city,state,city_ibge_code


### estimated_population

In [29]:
missing_estimated_population_mask = (dataset["estimated_population"].isna())

missing_estimated_population_lines = dataset.loc[
    missing_estimated_population_mask,
    ["estimated_population", "city", "state", "city_ibge_code"]
]

missing_estimated_population_lines

,estimated_population,city,state,city_ibge_code
16196,<NA>,Importados/Indefinidos,AL,<NA>
16200,<NA>,Importados/Indefinidos,AL,<NA>
16204,<NA>,Importados/Indefinidos,AL,<NA>
16208,<NA>,Importados/Indefinidos,AL,<NA>
16212,<NA>,Importados/Indefinidos,AL,<NA>
...,...,...,...,...
3754873,<NA>,Importados/Indefinidos,SP,<NA>
3755520,<NA>,Importados/Indefinidos,SP,<NA>
3756167,<NA>,Importados/Indefinidos,SP,<NA>
3756814,<NA>,Importados/Indefinidos,SP,<NA>


### city_ibge_code

In [30]:
missing_city_ibge_mask = (dataset["city_ibge_code"].isna()) | (dataset["city_ibge_code"] == "<NA>")

missing_city_ibge_list = dataset.loc[
    missing_city_ibge_mask,
    ["city", "state", "city_ibge_code"]
]

missing_city_ibge_list

,city,state,city_ibge_code
16196,Importados/Indefinidos,AL,<NA>
16200,Importados/Indefinidos,AL,<NA>
16204,Importados/Indefinidos,AL,<NA>
16208,Importados/Indefinidos,AL,<NA>
16212,Importados/Indefinidos,AL,<NA>
...,...,...,...
3754873,Importados/Indefinidos,SP,<NA>
3755520,Importados/Indefinidos,SP,<NA>
3756167,Importados/Indefinidos,SP,<NA>
3756814,Importados/Indefinidos,SP,<NA>


In [31]:
dataset.loc[
    dataset["last_available_confirmed_per_100k_inhabitants"].isna(),
    "estimated_population"
].describe()

count         15520.0
mean     10645.107023
std      80295.381959
min             776.0
25%            3308.0
50%            5341.0
75%           10560.0
max         4039277.0
Name: estimated_population, dtype: Float64